# 春学期第 3 回｜分布と欠損値

「この月の到着のずれは平均約 4 分」と聞くと，多くの便が 4 分遅れて着いたように感じるかもしれません。図と元の値から，早着した便の多さと長い遅れを読み分けます。中央の便，中央付近の広がり，60 分以上の遅れのどれを伝えたいかで，使う数を選びます。

最初に本文の例を一緒に実行し，後半では自分で条件を選んで約 20 分の実習に取り組みます。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のセルはファイルの置き場所に対応する準備です。そのまま実行してください。出力の `32347` は全予定便の行数です。到着のずれが記録された便数は，次の節で別に数えます。

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

file_name = "nyc-flights-2024-01.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

flights = pd.read_csv(csv_path)
len(flights)

## 本文と一緒に実行

各節の問いを読んでから，コードを上から順に実行します。図の棒が示す便と，平均などの計算に含まれる便の範囲を対応させてください。

### 到着のずれが記録された便を数える

2024 年 1 月の 3 空港を合わせた図です。高い棒は早着側にあるのに，平均の線は 0 分より右にあります。平均を「多くの便が着いた時刻」と読めるでしょうか。まず棒と線の位置を比べます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import pandas as pd
import matplotlib.pyplot as plt

opening_flights = flights
opening_delays = opening_flights["arrival_delay"].dropna()
opening_mean = opening_delays.mean()
opening_figure, opening_axis = plt.subplots(layout="constrained")
opening_axis.hist(opening_delays, bins=range(-90, 301, 10),
                  color="#286b70", edgecolor="white")
opening_axis.axvline(0, color="#494949", linestyle="--", linewidth=1.2,
                     label="Scheduled arrival")
opening_axis.axvline(opening_mean, color="#9c3d22", linewidth=1.8,
                     label=f"Mean: {opening_mean:.2f} min")
opening_axis.set(xlim=(-90, 300), xticks=[-90, -60, -30, 0, 60, 120, 180, 240, 300],
                 xlabel="Arrival delay (min)", ylabel="Flights")
opening_axis.spines[["top", "right"]].set_visible(False)
opening_axis.legend(frameon=False)
plt.show()

平均や中央値には，何分早い・遅いという値が必要です。全予定便から到着のずれが記録された便を取り出し，値がない便と，値はあるが図の横軸の外にある便を区別します。

In [ ]:
import pandas as pd

delay = flights["arrival_delay"]
observed_delays = delay.dropna()
print(len(flights), len(observed_delays), delay.isna().sum())

### 平均と中央値を 3 便から考える

元の表から 3 便を選び，符号付きのずれの合計を 3 で割った平均と，小さい順に並べた中央の値を求めます。この 3 便は計算を追う例で，月全体の代表ではありません。一つの遅れを大きくすると，二つの数がそれぞれどう動くかを考えます。

In [ ]:
example = flights.loc[[0, 82, 83], ["date", "origin", "destination", "arrival_delay"]]
example

In [ ]:
example_delays = example["arrival_delay"]
print(example_delays.sum(), len(example_delays), example_delays.mean())

In [ ]:
example_delays.sort_values()

In [ ]:
print(example_delays.mean(), example_delays.median())

3 便の例から月全体へ戻ります。早着便が半数を超えて中央値は負なのに，平均が正になるのはなぜでしょうか。早着の負の合計と遅着の正の合計を比べて説明します。

In [ ]:
print((observed_delays < 0).sum(), len(observed_delays))
print(observed_delays.mean(), observed_delays.median())

In [ ]:
early_delays = observed_delays.loc[observed_delays < 0]
late_delays = observed_delays.loc[observed_delays > 0]
print(len(early_delays), early_delays.sum())
print(len(late_delays), late_delays.sum())

### 足す値と割る件数

「到着のずれの平均」と「遅れた分の平均」は，同じ数とは限りません。早着の負の値を 0 にして遅着分を合計し，記録のある全便で割る場合と，遅着便だけで割る場合を比べます。合計と分母のどちらを変えたかを確認してください。

In [ ]:
positive_minutes = observed_delays.clip(lower=0)
positive_minutes.loc[[0, 82, 83]]

In [ ]:
print(positive_minutes.sum(), len(positive_minutes), positive_minutes.mean())

In [ ]:
print(late_delays.sum(), len(late_delays), late_delays.mean())

### 四分位数と長い遅れ

中央値だけでは，中央付近の便が何分間に広がるか分かりません。値を小さい順に並べた 25%，50%，75% の位置を求め，上段の順位に対応する下段の分を読みます。25% と 75% の間の幅も計算します。

In [ ]:
quartile_positions = [0.25, 0.5, 0.75]
monthly_quartiles = observed_delays.quantile(quartile_positions)
monthly_quartiles

In [ ]:
# 提供した作図セル。そのまま実行します。
from matplotlib.patches import ConnectionPatch

rank_figure, (rank_axis, value_axis) = plt.subplots(
    2, 1, layout="constrained"
)
rank_axis.set(xlim=(0, 1), xticks=[0, .25, .5, .75, 1],
              xticklabels=["0%", "25%", "50%", "75%", "100%"],
              xlabel="Position in sorted data")
rank_axis.xaxis.set_ticks_position("top")
rank_axis.xaxis.set_label_position("top")
value_axis.set(xlim=(-35, 25), xticks=[-23, -11, 10],
               xticklabels=["−23", "−11", "10"], xlabel="Arrival delay (min)")
for axis in (rank_axis, value_axis):
    axis.set_ylim(-1, 1)
    axis.set_yticks([])
    axis.spines[:].set_visible(False)
    axis.axhline(0, color="#666666", linewidth=1)
rank_axis.scatter(quartile_positions, [0] * 3, color="#286b70", s=48, zorder=3)
value_axis.scatter(monthly_quartiles, [0] * 3, color="#286b70", s=48, zorder=3)
for position, value in zip(quartile_positions, monthly_quartiles):
    link = ConnectionPatch(
        xyA=(position, 0), xyB=(value, 0), coordsA="data", coordsB="data",
        axesA=rank_axis, axesB=value_axis, color="#626b68",
        linestyle="--", linewidth=1, zorder=1,
    )
    rank_figure.add_artist(link)
plt.show()

ニューアーク（EWR）とラガーディア（LGA）の中央値と四分位数を比べます。中央の位置が早い空港では，60 分以上の遅れも少ないでしょうか。同じ軸の図で右側の棒を見てから，件数と割合を計算します。

In [ ]:
ewr_flights = flights.loc[flights["origin"] == "EWR"]
lga_flights = flights.loc[flights["origin"] == "LGA"]
ewr_delays = ewr_flights["arrival_delay"].dropna()
lga_delays = lga_flights["arrival_delay"].dropna()
print("EWR", len(ewr_flights), len(ewr_delays))
print("LGA", len(lga_flights), len(lga_delays))

In [ ]:
ewr_delays.quantile(quartile_positions)

In [ ]:
lga_delays.quantile(quartile_positions)

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

airport_figure, (ewr_axis, lga_axis) = plt.subplots(
    2, 1, sharex=True, sharey=True, layout="constrained"
)
bin_edges = range(-60, 181, 10)

# 上段の EWR
ewr_axis.axvspan(60, 180, color="#f3e5dd")
ewr_axis.hist(ewr_delays, bins=bin_edges, color="#286b70", edgecolor="white")
ewr_axis.axvline(0, color="#494949", linestyle="--", label="Scheduled arrival")
ewr_axis.axvline(ewr_delays.median(), color="#9c3d22", label="Median")
ewr_axis.axvline(60, color="#634638", linestyle=":", label="60 min or more")
ewr_axis.set(title="EWR", ylabel="Flights")
ewr_axis.legend(frameon=False)

# 下段の LGA
lga_axis.axvspan(60, 180, color="#f3e5dd")
lga_axis.hist(lga_delays, bins=bin_edges, color="#286b70", edgecolor="white")
lga_axis.axvline(0, color="#494949", linestyle="--")
lga_axis.axvline(lga_delays.median(), color="#9c3d22")
lga_axis.axvline(60, color="#634638", linestyle=":")
lga_axis.set(title="LGA", ylabel="Flights")

lga_axis.set(xlim=(-60, 180), xticks=[-60, -30, 0, 30, 60, 90, 120, 150, 180],
             xlabel="Arrival delay (min)")
plt.show()

60 分以上遅れた便を，図の 180 分より右にある便も含めて数えます。EWR と LGA で観測便数が違うため，それぞれの件数をその空港で到着のずれが記録された便数で割り，割合を比べます。

In [ ]:
ewr_long = ewr_delays >= 60
lga_long = lga_delays >= 60
print(ewr_long.sum(), len(ewr_delays))
print(lga_long.sum(), len(lga_delays))

In [ ]:
print(100 * ewr_long.sum() / len(ewr_delays))
print(100 * lga_long.sum() / len(lga_delays))

### 長い遅れの一行と空欄

最大の遅れがあった便を元の表で特定します。この 1 便だけで月全体の平均が正になったのでしょうか。仮にこの便を集計から外し，合計と便数の両方を変えた平均を元の平均と比べます。

In [ ]:
largest_label = observed_delays.idxmax()
flight_columns = ["date", "origin", "destination", "carrier",
                  "scheduled_departure", "arrival_delay"]
flights.loc[[largest_label], flight_columns]

In [ ]:
without_largest_total = observed_delays.sum() - observed_delays.max()
without_largest_count = len(observed_delays) - 1
without_largest_mean = without_largest_total / without_largest_count
print(without_largest_total, without_largest_count, without_largest_mean)

到着のずれが空欄の便について，欠航と到着先変更の記録を調べます。空欄を仮に 0 分と置くと，平均と中央値はどう変わるでしょうか。計算結果だけでなく，その 0 が実際の定刻到着を意味しないことも確認します。

In [ ]:
missing_flights = flights.loc[delay.isna()]
print(len(missing_flights))
print(missing_flights[["cancelled", "diverted"]].sum())

In [ ]:
filled_delays = delay.fillna(0)
print(observed_delays.sum(), len(observed_delays), observed_delays.mean())
print(filled_delays.sum(), len(filled_delays), filled_delays.mean())
print(observed_delays.median(), filled_delays.median())

In [ ]:
print((observed_delays == 0).sum(), (filled_delays == 0).sum())

## Jupyter 実習（約 20 分）

本文では EWR と LGA を比べました。今回は，2024 年 1 月の **JFK 発の便**について，到着のずれを伝える短い説明を作ります。

「中央の便は何分ほどずれたか」か「1 時間以上遅れた便はどれくらいか」の一方を選びます。どちらの問いにも同じ JFK の記録を使いますが，計算する数と図で見る場所は異なります。共通課題は 1〜3 の約 20 分です。末尾の追加課題は任意です。

### 1. 対象と図を読む（目安 5 分）

JFK 発の予定便を取り出し，到着のずれが記録された便を `observed` とします。予定便の件数と，値のある便の件数を別々に残します。`dropna()` は，この列で値が欠けた箇所を除いて計算用の列を作る操作です。元の予定便の表はそのまま残ります。

In [ ]:
airport_flights = flights.loc[flights["origin"] == "JFK"]
arrival_delay = airport_flights["arrival_delay"]
observed = arrival_delay.dropna()

print("予定便:", len(airport_flights))
print("到着のずれが記録された便:", len(observed))
print("値がない便:", arrival_delay.isna().sum())

予定便は 9,471，値がある便は 9,177，値がない便は 294 です。以下の図と要約は **値がある 9,177 便**を対象にします。

次の図は，本文の EWR・LGA の比較と同じく，横軸を −60 分から 180 分まで表示し，10 分幅で便を数えます。表示範囲の外にも記録があります。計算に使う `observed` からは，その記録を除いていません。

In [ ]:
fig, ax = plt.subplots()
ax.hist(observed, bins=range(-100, 1601, 10), color="#286b70", edgecolor="white")
ax.axvline(0, color="#834322")
ax.set(xlabel="Arrival delay (minutes)", ylabel="Flights", xlim=(-60, 180))
plt.show()

最も高い棒の区間と，0 分より右へ棒が続く範囲を読んでください。平均と中央値を表示し，二つの値が図のどこに位置するかを確かめます。

In [ ]:
print("平均（分）:", observed.mean())
print("中央値（分）:", observed.median())

**図と二つの値から気になったこと：**

（ここに書く）

### 2. 問いに合う要約を選ぶ（目安 9 分）

次の A と B から一つを選びます。図の中央付近を伝えたいか，60 分以上の遅れを伝えたいかで選んでください。選んだ方のセルだけ記入します。

#### A：中央の便は，どのくらいずれたか

中央値だけでなく，中央付近の約半数が入る範囲を調べます。`observed` に対して `quantile()` を呼び出し，`[0.25, 0.5, 0.75]` を渡す一行を書いてください。得られた値を `quartiles` に入れます。

下の 2 行の `#` を外し，1 行目の右側を補って実行します。

In [ ]:
# quartiles =
# quartiles

<details>
<summary>A の書き方を確認する</summary>

```python
quartiles = observed.quantile([0.25, 0.5, 0.75])
quartiles
```

左側の `0.25`，`0.50`，`0.75` は位置を表す割合で，右側がその位置の到着のずれ（分）です。結果は −24 分，−10 分，12 分です。負の値が早着を表すことを踏まえ，中央値と，中央付近の約半数が入る範囲を分単位で説明してください。

</details>

**A を選んだ人の記録：** 中央の位置／中央付近の約半数が入る範囲／範囲の幅（何分ぶんか）／図のどこに対応するか。

（ここに書く）

#### B：1 時間以上の遅れは，どのくらいあったか

`observed` が 60 分以上かを調べる比較を一行書き，`is_long` に入れます。下の 3 行の `#` を外し，1 行目の右側を補って実行します。件数を数えて割合に直す残りの 2 行は用意してあります。

分母は，図と同じく到着のずれが記録された便の件数です。

In [ ]:
# is_long =
# long_count = is_long.sum()
# print(long_count, len(observed), long_count / len(observed))

<details>
<summary>B の書き方を確認する</summary>

```python
is_long = observed >= 60
long_count = is_long.sum()
print(long_count, len(observed), long_count / len(observed))
```

出力は 844 便／9,177 便で，割合は約 0.092，百分率では約 9.2% です。図の横軸を 180 分で切っていても，この計算は 180 分より長い遅れを含みます。分母は予定便の全件数 9,471 便ではなく，到着のずれが記録された 9,177 便です。

</details>

**B を選んだ人の記録：** 60 分以上の件数／分母／割合／図のどこに対応するか。

（ここに書く）

### 3. 数と図から説明を書く（目安 6 分）

次の内容を含む短い段落を書いてください。A または B の一方だけで構いません。

1. 対象の時期・空港と，調べた問い。
2. 自分で出した要約値と，図のどの特徴に対応するか。
3. 平均だけ，または中央値だけを伝えた場合には，今回の問いについて何が分からないか。
4. 値がない 294 便を今回の計算でどう扱ったか。

**説明を書く欄：**

（ここに書く）

選んだ方のセルを記入し，ノートを最初から実行して保存します。選んでいない方のセルは，コメントのままで構いません。提出物は，自分の問い・実行した要約・上の説明が入ったノートブックです。

### 発展：処理を変えたら説明は変わるか（任意）

共通課題を終えたら，次のうち一つを試せます。

- 選ばなかった方の要約も求め，自分の説明へ一文だけ補う。
- JFK の最大値を確認し，その 1 件を除いた平均と元の平均を比べる。平均が変わったことだけで，その便を除いてよいと言えるか。
- `arrival_delay.fillna(0).mean()` と `observed.mean()` を比べる。値がない便を 0 分と置くことで，何を仮定したのか。差が小さい場合もその仮定が妥当になるとは限らない。

欠損の 0 補完を試す場合は，元の列を上書きせず，比較用の計算として行ってください。

In [ ]:
# 任意：追加で調べた操作と結果をここへ残す。